### Interference and Diffraction Lab 2024

#### Python routine to read position of Newport MM3000 motion controller and resulting current from Keithley 6485 ammeter.  Using the digital output of the ammeter helps reduce 60/120 Hz noise because the ammeter readings are synchronized to the power line frequency.  

###  We *strongly* suggest you run this Jupyter Notebook from the Windows Start menu Anaconda 3\Jupyter Notebook shortcut.  The shutdown procedure that frees up the USB resources has not been tested against Jupyter Navigator or against any random python notebook typed into the search box which could launch you into a bad place.  When you are done with this Notebook, please be kind to the next lab group:  close this notebook, then navigate to the black and white Jupyter Notebook server in the Task Bar and type Ctrl-C twice. 

### To get the best symmetry on the results, remove the aperture or knife edge by loosening the vertical post.  Now loosen the bolts holding the base of the detector and move it carefully so that the expanded laser beam hits the center of the detector.  Tighten.  If your diffraction pattern looks lopsided, check this again.  Put the aperture back in place.  Make sure that the black arrows on the translation stage are aligned (use blue buttons on motion controller (for fast jog, hold down center button and press either arrow) to bring it back to center as necessary, then press Reset on controller to set this current position as Home, 0.0000).  Positive motion is away from the power cord.  Now unlock the large dovetail slide beneath the translation stage to center the diffraction pattern on the PMT slit, then retighten.  If the pattern is strongest in the middle, you can put the ammeter into Local mode and adjust the position of the slit to maximize the reading.




********************************************
PJHT 2 August 2022

Alex Hiser port from LabView 26 July 2022

PJHT cleanup, autoscaling, convolution, Pythonic min/max, file retrieval 3-16 April 2023 and January 2024

James Zabel adapted to use GrinnellHardwareDetection
********************************************
This notebook can be run from any Windows PC that has NI-VISA and pyvisa installed

Cabling:  Prolific USB A to DE9 RS232 converter from PC, then straight-through DE9 cable to Newport MM3000 motion controller
          Prolific USB convert from USB A on PC to DE9 RS232 cable to Keithley 6485 picoammeter.  Set Keithley at 9600, N, 8, 1.  
          
Computer setup:  Assumes NI-VISA is installed from ni.com

In Python commmand line, make sure that pyvisa is installed.  Also install pyautogui and keyboard as well.


          
### Initialization -- run this once:
#Setup block. Run this once to import the libraries and files we will need.  Use `%matplotlib` qt magic to get external plot figures for animation, which gives live plotting of data.  

The magic command appears *before* `import matplotlib.pyplot as plt`

`%matplotlib inline` won't work with `FuncAnimation()`

`%matplotlib notebook` will work with `FuncAnimation()`, but cannot interact with keyboard

`%matplotlib qt` is best because you get full use of keyboard.  Plots are external, appearing in taskbar




In [1]:
%matplotlib qt

import matplotlib.pyplot as plt                   # for plotting
from matplotlib.animation import FuncAnimation    # for live plotting
import numpy as np                                # for doing math and working with arrays
from time import time, sleep, localtime, strftime # some timekeeping functions
import scipy.special as ss                        # for plotting Fresnel Integrals
import ast                                        # abstract syntax tree used to parse the dictionary
import scipy.integrate as integrate               # for solving Fresnel Integrals manually
from scipy import signal                          # for convolving data with PMT aperture to model 
                                                  #   finite spatial resolution of detector

try:
    import pyvisa as visa
except:
    print("\nIt appears that pyvisa is not available.  Install via an Anaconda Prompt")
    print("by using >pip install pyvisa (Requires a working network connection and admin privileges)\n")
try:
    import keyboard # needed to shut down and close resources and file handles
except:
    print("\nIt appears that keyboard is not available.  Install via an Anaconda Prompt")
    print("by using >pip install keyboard (Requires a working network connection and admin privileges)\n")
try:
    import pyautogui
except:
    print("\nIt appears that pyautogui is not available.  Install via an Anaconda Prompt")
    print("by using >pip install pyautogui (Requires a working network connection and admin privileges)\n")

from GrinnellHardwareDetection import *

import ipywidgets as widgets  # for buttons
import os                     # to remove files
import sys                    # for sys.exit
import winsound               # to beep on failure

button_layout = widgets.Layout(width = "auto", height = "50px")
removeDataButton = widgets.Button(description = "Click here to delete these data!", layout = button_layout)
out = widgets.Output()

def removeData(_):
    filename = expt_params['filename']  # get the filename from the current dictionary
    print("\nDeleting data... file = ", filename)
    os.remove(filename)
    removeDataButton._click_handlers.callbacks = [] # make the button unusable

DEBUG = False # hides/shows debug print statements

print("You may proceed to the next cell!")

You may proceed to the next cell!


### A1: Establish communications with both instruments

This is done via PyVISA which abstracts the devices as "resources". It queries each resource to determine if it is needed for this experiment and assigns it appropriately.

After you run this cell successfully you need not return here.

In [ ]:
rm = visa.ResourceManager() # open the resource manager as rm
print("Here are the available resources (USB ports): ")
print(rm.list_resources())
resources = rm.list_resources()

res_list = ['ASRL1::INSTR'] # we won't be using this, so we put it as 'already taken' in resource list

motion_controller = get_motion_controller(resources, res_list, rm, DEBUG)

needed_devices = [ "MODEL 6485" ]
found_devices  = connect_to_Grinnell_Hardware(resources, res_list, rm, needed_devices, DEBUG)

ammeter = found_devices[0] # Keithley 6485

print("You may proceed to the next cell!")

### A2: Configure the devices

Each device needs to be properly configured for the needs of this experiment:
* The Keithley 6485 needs to have the current range set to autoranging, to be set for slow or medium instrument response, to have autozero turned off, and to have zero check turned off.
* The Motion Controller has a number of fixed settings to optimize its motion.  There is no need to adjust these.

After you run this cell successfully you need not return here.

In [ ]:
# A2:  Configure the devices
# reopen the resources
motion_controller.open()
motion_controller.timeout = 20000
ammeter.open()

# set up the Keithley 6485 (ammeter)
print("Resetting and configuring the Keithley 6485 (used as an ammeter to measure the current produced by the PMT)...")
ammeter.write("*CLS")                      # clear errors
ammeter.write("*RST")                      # reset
sleep(3)
ammeter.write(":DISP:TEXT:STAT ON")        # allows text to be displayed on the screen
ammeter.write(":DISP:TEXT:DATA 'Ammeter'") # display this devices's role
ammeter.write(":SENS:CURR:RANG:AUTO ON")   # turn on autoranging
ammeter.write("CURR:NPLC 1")               # SLOW instrument response (must be greater than or equal to 1 or 
                                           #   you get aliasing with 120 Hz room lights).  If you make this 6, you may need
                                           #   to scan more slowly by selecting an even smaller value than 0.01 in the animation
                                           #   loop below:  motion_controller.write("1UV0.01") # set speed to 0.01 mm/s 
ammeter.write(":SYST:ZCH ON")              # turn on zero check
ammeter.write(":SYST:AZER:STAT OFF")       # turn off autozero
ammeter.write(":FORM:ELEM READ")           # read out only the current reading, not other info
ammeter.write(":SYST:ZCH OFF")             # turn off zero check to be ready to read
ammeter.write(":SYST:ZCOR OFF")            # turn off zero correction
print("\nThe Keithley 6485 measuring current from the PMT is initialized!")

print("   Test that we can read the ammeter:")
current = -1.0 * float(ammeter.query(":READ?"))
print("     The ammeter reads: {:4.3e}".format(current))

# set up the Motion Controller
# Initialize motion controller assuming the use of MFN25CC Motor: All values found in the Appendix (p. 8.19)
print("Configuring the Motion Controller...")
motion_controller.write('1US0.0555um')
motion_controller.write('1UUmm')
motion_controller.write('1UV0.3')  # this is for high speed motion, setting the peak velocity
motion_controller.write('1UA1.2')  # this sets the acceleration of the motor
motion_controller.write('1FE+500')
motion_controller.write('1IL+100')
motion_controller.write('1KP+1500')
motion_controller.write('1KD+5000')
motion_controller.write('1KI+100')
print("\nThe Motion Controller is initialized!")

# close the resources
motion_controller.close()
ammeter.close()

print("Configuration complete... You may proceed to the next cell!")


In [8]:
# A3:  Functions needed to run the experiment.  Run this once before taking data.  No need to revisit later

def StepsToMM(steps): # convert from motor steps to mm
    return (float(steps) * 0.0000555) # (Each step moves the stage by 0.0000555 mm.  Do not change this parameter)

def MoveToStart(): # move to the starting position
    motion_controller.open()
    motion_controller.write('1UV0.3') # moving at high speed
    if(start_pos_x < 0):
        motion_controller.write(f'1UP-{abs(start_pos_x / 1000)}')
        #motion_controller.write("1UP -{:6.4f}".format(abs(start_pos_x / 1000)) )# modern format in mm (how many places?)
    else:
        motion_controller.write(f'1UP+{abs(start_pos_x / 1000)}')

    present_position = 10000 # an impossibly large number, just to make sure that the next loop runs

#  while(abs(start_pos_x - StepsToMM(int(str(motion_controller.query('1TPI')).rsplit(' ')[0])) * 1000) > 10):                                    
    while(abs(start_pos_x - present_position) > 10.0):
        # Wait until starting position has been reached
        sleep(1)
        present_position = StepsToMM(int(str(motion_controller.query('1TPI')).rsplit(' ')[0]))*1000 
        #print(present_position)
        continue

def init_expt(expt_params):
    # output file setup
    #  First line is the dictionary
    ammeter.open()
    filename = "Fresnel_Diffraction_" + diffraction_type + "_" + strftime("%d_%b_%y_%I-%M-%S", localtime()) + ".csv"
    expt_params['filename'] = filename  # add it to the dictionary
    f_out = open(filename, 'w')
    # write the dictionary to the top of the file
    f_out.write(str(expt_params)+"\n")
#     f_out = open(filename, 'w', newline = '')
#     f_out
#     f_out.write("Fresnel Diffraction for " + diffraction_string + " with R_s = {:4.3f} and R_o = {:4.3f}\n".format(r_s, r_o))
    
    f_out.write("position (mm), current (A)\n")  # write the column headers
    print("Output file "+filename+" is open for writing")

    return f_out, expt_params  # return the output file handle and the updated dictionary


def get_plot_style(diffraction_type):
    
    if(diffraction_type == "Knife_Edge"):
        diffraction_string = "Knife Edge"
        plot_style = "-b" # blue
    elif(diffraction_type == "Single_Slit"):
        diffraction_string = "Single Slit"
        plot_style = "-g" # green
    elif(diffraction_type == "Double_Slit"):
        diffraction_string = "Double Slit"  
        plot_style = "-p" # purple

    print(diffraction_string)
    return diffraction_string, plot_style

     
def update(count, done_flag, ): # function to update the live plot and save the data during the animation  
       
    while(done_flag == False):
        elapsed_time = time() - start_time
       
        current = -1.0 * float(ammeter.query(":READ?"))  # read current first (accounts for autoscaling delay)
        
        position = StepsToMM(int(str(motion_controller.query("1TPI")).rsplit(' ')[0]))
        position_data.append(position)       
        current_data.append(current)
        remaining_time = elapsed_time / ((position * 1000 - start_pos_x) / (end_pos_x - start_pos_x)) - elapsed_time
        #print(position, current)
        ln.set_data(position_data, current_data)  
        ax.relim()
        ax.autoscale_view()
        count += 1
        pyautogui.press('f15') # keep alive
        f_out.write("{:5.3f}, {:6.3e}\n".format(position, current))
#         print("Current Position: {:5.2f} mm, Elapsed Time: {:5.2f} sec, Est. Remaining Time: {:5.2f} sec".\
#              format(position, elapsed_time, remaining_time), end=" \r")
        if(position * 1000 >= (end_pos_x-2)): # final position has been reached (or close enough)
            done_flag = True
            ani.event_source.stop()
            sleep(1)
            close_resources()
            sleep(3)  # need to wait at least two seconds before making static plot
            generate_static_plot(filename)
        else:
            return ln,

def press(event): # this function gracefully stops the experimental apparatus when halting during data collection
    if(event.key == 'q'):
        print("\nYou pressed 'q' so the scan will now stop and return to home position...")
        motion_controller.write('#') # abort motion
        close_resources()
        generate_static_plot()
        print("All individual resources and file handle are closed.")

def close_resources(): # function to close all resources
    ammeter.write(":DISP:TEXT:STAT ON")        # allows text to be displayed on the screen
    ammeter.write(":DISP:TEXT:DATA 'Ammeter'") # text to display on the screen
    ammeter.close()
    motion_controller.write('1UV0.3') # moving stage back to Home position at high speed
    motion_controller.write('1UA1.2')
    motion_controller.write('1UP0.0000')
    motion_controller.close()
    f_out.close()

def generate_static_plot(filename, style = "-b"): # function to generate a static plot
    fig, ax = plt.subplots(1, 1, figsize = (12,8), constrained_layout = True)
    plt.suptitle("Fresnel Diffraction for a " + diffraction_string + ", file = {:}".format(filename))
    ax.set_title("Current vs. Position in lab units")
    ax.set_xlabel("Position (mm)")
    ax.set_ylabel("Current (A)")
    ax.set_xlim(position_data[0], position_data[-1])
    ax.grid()
    ax.plot(position_data, current_data, style)
    plt.show()

print("\nYou may proceed to the next cell!")


You may proceed to the next cell!


### A3: Define the functions and experimental parameters used for data collection

**You need to do a little work here.**  

For each experiment that you do, record the type of slit (allowed names:  **Knife_Edge**, **Single_Slit**, **Double_Slit**), the slit parameters (slit width **a_lab** and slit spacing **d_lab**, in mm) the source-aperture **r_s** and aperture-observer (PMT slit) **r_o** distances in meters.  Then pick a starting and ending position (in microns) for the scan.  Edit the dictionary below as needed, or at the top of each experimental scan.  To clear out an unneeded key, such as **'d_lab'**, type for example, **'d_lab' = []** In setting up the experimental parameters, you should edit values in the dictionary of parameters **expt_params** and then read the parameters back from the dictionary.  By keeping the dictionary up to date, we make sure that the top line of the output data file contains everything you need to know to replicate the experiment, or to go back and reanalyze your data using Kirchoff scalar diffraction theory.

You'll see that we first set up the dictionary:

In [20]:
# A4: Define a dictionary to hold most experimental parameters.  This will simplify passing arguments to functions.
#   Edit as needed.   This example is for the Double Slit experiment, which requires the most parameters:
expt_params = {'title': "Fresnel Diffraction", # Indentifier of experiment
               'r_s' : 0.170,          # source pinhole to aperture distance in meters
               'r_o' : 0.473,          # observation slit to aperture distance in meters
               'lamb' : 632.8,         # wavelength in nm
               'a_lab' : 0.40,         # slit width in mm
               'd_lab' : 1.20,         # slit spacing in mm
                                       # The motion controller has an absolute limit of 25.4 mm travel, which is 
                                       #    25,400 microns.  We typically need only +/- 2000 or 3000 microns
               'start_pos_x' : -1000,  # start position of scan, in micrometers, typically -2000 for single and double slits
                                       #   and maybe -1000 for knife edge
               'end_pos_x' : 1000,     # end position of scan, usually 2000 for slits, 3000 for knife edge.  Must be 
                                       #  must be greater than start_pos_x
               'diffraction_type' : "Double_Slit",
               'pmt_slit_width' : 25,  # this is fixed at 25 microns
               'filename' : []         # placeholder for output filename, populated at run time
              }

def print_dictionary(expt_params):
    print("These are the current dictionary values: ")
    for p, v in expt_params.items():
        print("{:20}\t{:}".format(p, v))
    if (expt_params['start_pos_x'] >= expt_params['end_pos_x']):
        print("**Error:  end_pos_x must be greater than start_pos_x**")
    if (expt_params['start_pos_x'] < -10000):
        print("**Error:  you are likely to stall out the slide in the negative direction [x < -10000]**")
    if (expt_params['end_pos_x'] > 10000):
        print("**Error:  you are likely to stall out the slide in the positive direction [x > 10000]**")
      
    print("\n")   
    
print_dictionary(expt_params) # execute this line whenever you want to see what values are in the dictionary

These are the current dictionary values: 
title               	Fresnel Diffraction
r_s                 	0.17
r_o                 	0.473
lamb                	632.8
a_lab               	0.4
d_lab               	1.2
start_pos_x         	-1000
end_pos_x           	1000
diffraction_type    	Double_Slit
pmt_slit_width      	25
filename            	[]




### All the remaining sections of this notebook are tailored for knife edge (B1), single slit (B2), and double slit (B3).  Data are acquired in the same way, but we divide these sections up to make sure that you stop and remeasure r_o and r_s, and get the correct a_lab and d_lab as needed for *each* experimental run. At any time after taking data, go down to section C to analyze with Kirchoff scalar diffraction theory.

### B1:  Data Acquisition for Knife Edge Diffraction

In [ ]:
#  B1:  First, print the dictionary.  Don't worry about the filename:  that will be updated when the experiment starts
print_dictionary(expt_params)

###  B1, continued:  Place the knife edge in its holder so that it points away from you, with the knife edge facing the end of the motor with the cable, and center it so that the pattern is aligned with the slit on the front of the PMT.  Use the cell below to update any dictionary parameters, and then print them out before you start the experiment. Because the knife edge needs to go far away from the center to see the whole effect (while capturing the shadow region), we recommend starting at around -1000 microns and scanning to 3000 microns. 

In [ ]:
# B1, continued:  Data acquisition for knife edge
#  Declare the type of aperture you are using
diffraction_type = "Knife_Edge"  # Setting this now to knife edge.  Possible choices are "Knife_Edge", "Single_Slit",
                                 #   or "Double_Slit"

diffraction_string, plot_style = get_plot_style(diffraction_type)  # update plot header
# Now edit the dictionary as needed:
expt_params['diffraction_type'] = diffraction_type  # this writes "Knife_Edge" to the dictionary)
expt_params['a_lab'] = []  # no slit width for knife edge so clear it out
expt_params['d_lab'] = []  # no slit spacing for knife edge so clear it out

print_dictionary(expt_params)  # Show the contents of the dictionary

#  Get the current values of starting and ending positions from the dictionary

#  Here's how you change a parameter in the dictionary:
expt_params['start_pos_x'] = -1000
#  You'll want to do the same for 'end_pos_x',  'r_s', and 'r_o'
expt_params['end_pos_x'] = 3000
expt_params['r_s'] = .240  #  this is the pinhole source to knife edge distance in meters
expt_params['r_o'] = .245  #  this is the knife edge to PMT slit distance in meters
# Review these and change as needed and rerun this cell until dictionary is up to date:
print_dictionary(expt_params)




In [ ]:
# B1, continued:  Now extract the needed values from the dictionary and start the experiment.  The dictionary will
# form the first line in the data file, so it is important to keep it up to date
r_o = expt_params['r_o'] ### # knife edge to PMT slit spacing, in meters
r_s = expt_params['r_s'] ### # spatial filter pinhole to knife edge spacing, in meters
start_pos_x = expt_params['start_pos_x'] # μm  # recommended:  scan from -1000 to 3000 for knife edge
end_pos_x   = expt_params['end_pos_x']  # μm

fig, ax = plt.subplots(1, 1, figsize = (12,8), constrained_layout = True)
plt.suptitle("To quit early, hover over plot and press q.")
ax.set_title("Fresnel Diffraction for a " + diffraction_string)
ax.set_xlabel("Position (mm)")
ax.set_ylabel("Current (A)")
ax.grid()
ax.set_xlim(start_pos_x / 1000, end_pos_x / 1000)    # scale the plot, in mm
position_data, current_data = [], []  # empty data lists
ln, = plt.plot([], [], plot_style) # this creates the ln (line) tuple with empty lists
ln.set_data(position_data, current_data) # this tells ln to look at position_data, current_data
done_flag = False

start_time = float(time())
f_out, expt_params = init_expt(expt_params)  # get updated dictionary, which includes the filename
print("Moving to starting position...") # issue the scan to command and get going:
print("Live plot may appear in the task bar at bottom of screen")

MoveToStart() #  move at high speed
count = 0
motion_controller.write("1UV0.01") # set speed to 0.01 mm/s (for slow scan)
print("Starting the scan")
motion_controller.write(f'1UP+{abs(end_pos_x / 1000)}') # start the motion
# The plot update rate is set by 'interval' which is in milliseconds.  We use 20 or 50.
ani = FuncAnimation(fig, update, interval = 20, cache_frame_data = False, fargs = [done_flag])

cid = fig.canvas.mpl_connect('key_press_event', press)
plt.show()

removeDataButton.on_click(removeData)
buttons = widgets.HBox([removeDataButton])
display(widgets.VBox([buttons, out]))

print("When you have finished taking data, scroll down to Cell C1 to plot a theoretical model atop it")

### B2:  Data Acquisition for Single Slit Diffraction
####  Carefully remove the knife edge and replace with a single slit.  Use the appropriate cardboard mask to select which of the three slits you want, and update the expt['a_lab'] value according to the value in mm posted on the inside of the door.  Typically a symmetrical scan from -2000 to 2000 is needed here.

In [10]:
#  B2:  First, print the dictionary.  Don't worry about the filename:  that will be updated when the experiment starts
print_dictionary(expt_params)

These are the current dictionary values: 
title               	Fresnel Diffraction
r_s                 	0.17
r_o                 	0.473
lamb                	632.8
a_lab               	0.4
d_lab               	1.2
start_pos_x         	-1000
end_pos_x           	1000
diffraction_type    	Double_Slit
pmt_slit_width      	25
filename            	[]




In [12]:
# B2:  Edit the dictionary here:
diffraction_type = "Single_Slit"  # Setting this now to single
diffraction_string, plot_style = get_plot_style(diffraction_type)  # update plot header
expt_params['diffraction_type'] = diffraction_type  # write it to the dictionary

expt_params['d_lab'] = []  # no slit spacing for single slit so clear it out
expt_params['a_lab'] = 0.40  # change as needed
expt_params['start_pos_x'] = -1000 ### change as needed
expt_params['end_pos_x'] = 1000 ### change as needed
print_dictionary(expt_params)

Single Slit
These are the current dictionary values: 
title               	Fresnel Diffraction
r_s                 	0.17
r_o                 	0.473
lamb                	632.8
a_lab               	0.4
d_lab               	[]
start_pos_x         	-1000
end_pos_x           	1000
diffraction_type    	Single_Slit
pmt_slit_width      	25
filename            	[]




In [14]:
# B2, continued:  Now extract the needed values from the dictionary and start the experiment.  The dictionary will 
#   form the first line in the data file, so it is important to keep it up to date 

#  Get the current values of starting and ending positions from the dictionary
start_pos_x = expt_params['start_pos_x'] # μm
end_pos_x   = expt_params['end_pos_x']  # μm
#  You will need to update r_s and r_o in a similar fashion
#  Did r_s and r_s change?  Update the disctionary here:
# expt_params['r_s'] = ###
# expt_params['r_o'] = ###
# expt_params['a_lab'] =

fig, ax = plt.subplots(1, 1, figsize = (12,8), constrained_layout = True)
plt.suptitle("To quit early, hover over plot and press q.")
ax.set_title("Fresnel Diffraction for a " + diffraction_string)
ax.set_xlabel("Position (mm)")
ax.set_ylabel("Current (A)")
ax.grid()
ax.set_xlim(start_pos_x / 1000, end_pos_x / 1000)    # scale the plot, in mm
position_data, current_data = [], []  # empty data lists
ln, = plt.plot([], [], plot_style)
ln.set_data(position_data, current_data) 
done_flag = False

start_time = float(time())
f_out, expt_params = init_expt(expt_params)  # get updated dictionary, which includes the filename
print("Moving to starting position...") # issue the scan to command and get going:
print("Live plot may appear in the task bar at bottom of screen")

MoveToStart() #  move at default high speed
count = 0
motion_controller.write("1UV0.01") # set speed to 0.01 mm/s (for slow scan)
print("Starting the scan")
motion_controller.write(f'1UP+{abs(end_pos_x / 1000)}') # start the motion

ani = FuncAnimation(fig, update, interval = 20, cache_frame_data = False, fargs = [done_flag])

cid = fig.canvas.mpl_connect('key_press_event', press)

plt.show()

removeDataButton.on_click(removeData)
buttons = widgets.HBox([removeDataButton])
display(widgets.VBox([buttons, out]))

print("When you have finished taking data, scroll down to Cell C1 to plot a theoretical model atop it")

NameError: name 'ammeter' is not defined

### B3:  Data Acquisition for Double Slit Diffraction
#### Replace with a double slit. Use the appropriate cardboard mask to select which of the three pairs of slits you want, and update the expt_params['a_lab'] value and expt_params['d_lab'] according to the values in mm posted on the inside of the door.  Typically a symmetrical scan from -2000 to 2000 is needed here.

In [ ]:
# B3: editing the dictionary
print_dictionary(expt_params)

diffraction_type = "Double_Slit"  # Setting this now to double
diffraction_string, plot_style = get_plot_style(diffraction_type)  # update plot header
#  Edit the dictionary to give the current values of starting and ending positions
expt_params['diffraction_type'] = diffraction_type
expt_params['r_o'] = 0.880
expt_params['r_s'] = 0.19
start_pos_x = expt_params['start_pos_x'] # μm
end_pos_x   = expt_params['end_pos_x']  # μm
print_dictionary(expt_params)
#  You may need to update r_s and r_o in a similar fashion


In [ ]:
# B3, continued:  Now extract the needed values from the dictionary and start the experiment.  The dictionary will 
# form the first line in the data file, so it is important to keep it up to date
fig, ax = plt.subplots(1, 1, figsize = (12,8), constrained_layout = True)
plt.suptitle("To quit early, hover over plot and press q.")
ax.set_title("Fresnel Diffraction for a " + diffraction_string)
ax.set_xlabel("Position (mm)")
ax.set_ylabel("Current (A)")
ax.grid()
ax.set_xlim(start_pos_x / 1000, end_pos_x / 1000)    # scale the plot, in mm
position_data, current_data = [], []  # empty data lists
ln, = plt.plot([], [], plot_style)
ln.set_data(position_data, current_data) 
done_flag = False

start_time = float(time())
f_out, expt_params = init_expt(expt_params)  # get updated dictionary, which includes the filename
print("Moving to starting position...") # issue the scan to command and get going:
print("Live plot may appear in the task bar at bottom of screen")

MoveToStart() #  move at default high speed
count = 0
motion_controller.write("1UV0.01") # set speed to 0.01 mm/s (for slow scan)
print("Starting the scan")
motion_controller.write(f'1UP+{abs(end_pos_x / 1000)}') # start the motion

ani = FuncAnimation(fig, update, interval = 20, cache_frame_data = False, fargs = [done_flag])

cid = fig.canvas.mpl_connect('key_press_event', press)

plt.show()

removeDataButton.on_click(removeData)
buttons = widgets.HBox([removeDataButton])
display(widgets.VBox([buttons, out]))
print("When you have finished taking data, scroll down to Cell C1 to plot a theoretical model atop it")

###  C1:  Compare data with theory for any type aperture.  Read in and plot the data.  It reads in the dictionary to determine the type of diffraction aperture and relevant experimental parameters.

### In computing the Kirchoff scalar diffraction model, it assumes the nominal values of slit width and spacing as given by the manufacturer and recorded in your dictionary.  You may need to make small adjustments (< $\pm 10 \% $) to these for best fit.  If your data are lopsided, double-check the centering of the expanded laser beam and reposition the detector as needed.

###  Vertically, the data and model for the apertures are are scaled to the maximum height of the data; horizontally, the center of the pattern is deduced from the center of mass (aka centroid) of the data for the single and double slits, and the data are then shifted to place the center of the pattern at the dimensionless u = 0.00 origin.  This latter procedure should work well if the pattern is centered at the zero (Home) position. For the knife edge, the vertical scaling is determined from the limiting intensity.  The horizontal scaling is determined by finding the limiting intensity of the data, then shifting the pattern back until data cross  the 1/4 Intensity point, famously at the edge of the geometrical shadow.

### In a second plot, the model is convolved with an instrument width that assumes a 25 micron slit, to account for the smearing (finite precision) of the data.  As usual, the physical units are converted to dimensionless *u* units for comparison.  The actual instrument width will depend on how well you align the vertical diffraction pattern with the slit, as well as the scanning and sampling rates.

### Finally, an optional model in section C2 accounts for the spatial falloff in the laser intensity, because the expanded beam has a Gaussian beam profile, rather than a pure spherical wavefront.

In [30]:
a_lab = 0.40
a_lab

0.4

In [52]:
# C1: Compare theory and data for all aperture types by reading in the most recent file (the one in the dictionary).  
filename = 'Fresnel_Diffraction_Single_Slit_Complement_15_Apr_26_01-47-44 (1).csv'  # This code cell defaults to the present filename.  Edit as needed
                                    #   to pull up and reanalyze a previous data set
#   If you want to call up an old file, simply uncomment the first line and put your file name into the dictionary 
# expt_params['filename'] = 'Fresnel_Diffraction_Knife_Edge_18_Jan_24_01-09-07.csv'  
# (For simplicity, I run this notebook from my desktop, and so the data files get stored in the same directory, and
#   I don't need to specify the full path name.  If you run this notebook from your default user directory, you'll need to 
#   specify the full path name, something like:

#    filename = "C:/Users/tjossem.GRIN.001/Desktop/Fresnel_Diffraction_Knife_Edge_18_Jan_24_01-09-07.csv"

#   Of course, that file no longer exists because ITS has been wiping out all user documents and desktop files after 10 days of
# not logging in....

# One more important aspect of file names in Windows.  Windows thinks the path to the file is 
#   C:\Users\tjossem.GRIN.001\Desktop\Fresnel_Diffraction_Knife_Edge_18_Jan_24_01-09-07.csv
# but for our purposes we'll replace all the forward slashes with backslashes when handling the string in Python.  That's
# because \t \n \r are all interpreted as potential Escape sequences (formatting instructions in C/C++).  So to indicate that 
# they are not Escape sequences, just reverse the \ to /. (Alternatively, replace all single slashes with double slashes,
# indicating that they are 'literals' and not parts of an Escape sequence.)
    
print("Opening file ", filename)
print("Extracting the dictionary of experimental parameters: \n")

with open(filename,'r')as f_in:
    dict_string  = f_in.readline()
    header = f_in.readline()  # get the header as well
expt_params = ast.literal_eval(dict_string)
print_dictionary(expt_params) # print the dictionary to determine what type of model is needed
#  Get diffraction type from dictionary:
diffraction_type = expt_params['diffraction_type']

diffraction_string, plot_style = get_plot_style(diffraction_type)  # update plot header
# Get necessary parameters from the dictionary to calculate model:
r_o = expt_params['r_o']
r_s = expt_params['r_s']
lamb = expt_params['lamb']
a_lab = 0.4
d_lab = expt_params['d_lab']
pmt_slit_width = expt_params['pmt_slit_width'] # get the width of the slit at the PMT for convolution

#  Compute the scale factor that maps lab distances onto dimensionless aperture coordinates:

scale_factor = np.sqrt((2 / (lamb*1.0e-9)) * (1 / r_s + 1 / r_o)) # Elmore and Heald Eq 11.2.12

# Now read in the header and the data:
print(header)
# Read in data using loadtxt, skipping the first two rows that contain the dictionary and the header
position_data, current_data = np.loadtxt(filename,
    delimiter = ",", skiprows = 2, unpack = True) 
position_arr = np.array(position_data)  # convert the text data into (x,y) floating point arrays
current_arr = np.array(current_data)

# In comparing the data to a model, we choose to convert the data to the normalized u coordinates used in the model:
u_exp = position_arr/(1000) * scale_factor   # convert to u units 
#  The model itself will be computed on a fine grid so that it will plot as a smooth, not chunky, line:
u_mod = np.linspace(-5,5,20000)  # fine grid, 10000 points over 20 u units


#  Compute the model from Kirchoff scalar diffraction theory, depending on the aperture type:
if (diffraction_type == "Knife_Edge"):
    y_mod = np.square(ss.fresnel(u_mod)[1] - (-0.5)) + np.square(ss.fresnel(u_mod)[0] - (-0.5))


elif (diffraction_type == "Single_Slit"):
    a_lab_in_meters = a_lab/1000.0
    a_scaled = scale_factor * a_lab_in_meters  # scale the lab distances in order to put into dimensionless u coordinates
    #  Now the diffraction integral integrates from the left edge to the right edge of the single slit
    y_mod = np.square(1-ss.fresnel(u_mod + a_scaled/2)[1] + ss.fresnel(u_mod - a_scaled/2)[1]) +\
        np.square(1-ss.fresnel(u_mod + a_scaled/2)[0] + ss.fresnel(u_mod - a_scaled/2)[0])
    print("Physical slit width: {:5.3f} mm".format(a_lab))
    print("Dimensionless slit width {:5.4f}".format(a_scaled))


elif (diffraction_type == "Double_Slit"):
    a_lab_in_meters = expt_params['a_lab']/1000.0
    d_lab_in_meters = expt_params['d_lab']/1000.0
    a_scaled = scale_factor * a_lab_in_meters
    d_scaled = scale_factor * d_lab_in_meters
    #  Now the diffraction integral integrates from the left edge to the right edge of each slit
    y_mod = np.square(ss.fresnel(u_mod + d_scaled/2 + a_scaled/2)[1] - ss.fresnel(u_mod + d_scaled/2 - a_scaled/2)[1] + \
                      ss.fresnel(u_mod - d_scaled/2 + a_scaled/2)[1]\
                   - ss.fresnel(u_mod - d_scaled/2 - a_scaled/2)[1])\
    + np.square(ss.fresnel(u_mod + d_scaled/2 + a_scaled/2)[0] - ss.fresnel(u_mod + d_scaled/2 - a_scaled/2)[0] 
             + ss.fresnel(u_mod - d_scaled/2 + a_scaled/2)[0] - ss.fresnel(u_mod - d_scaled/2 - a_scaled/2)[0])
    print("Physical slit width: {:5.3f} mm and slit spacing {:5.3f} mm".format(a_lab, d_lab))
    print("Dimensionless slit width {:5.4f} and slit spacing {:5.4f}".format(a_scaled, d_scaled))

# Now clean up the data a little if necessary.  This step is optional, and is probably not needed.  It averages 
#   the first and last 10 data points in order to get an estimate of the baseline
#baseline = np.sum(current_arr[0:10]+current_arr[-10:])/20
# For now, assume the baseline is zero:
baseline = 0.00  # For really wide patterns you may not get a good baseline here.  If you keep the room lights off
                 #   you should find that the baseline is essentially zero and so for now we fix it that way
current_arr=current_arr-baseline  # optional vertical adjustment

# Now find the center of the pattern (centroid) by doing a 'center of mass' calculation, which multiplies
#   the signal times its position, then divides by the sum of all the signals.

center_of_mass = (current_arr*position_arr).sum()/current_arr.sum()  # centroid
print("center of mass (centroid) of data: ", center_of_mass)
position_arr = position_arr-center_of_mass    # subtract the centroid so as to center the data on the theory

#  Now that we have the model, do some scaling vertically so as to normalize the arbitary intensity.  For the single
#    and double slits, we simply match the vertical heights:

if (diffraction_type != "Knife_Edge"):    
    max_val_mod = y_mod.max()  # find max value of model to use for vertical scaling
    max_val = current_arr.max()  # find max value of data to use for vertical scaling
    y_mod = y_mod*max_val/max_val_mod  # scale the model to match the height of the data

else:  #  Here for the knife edge
#  A remarkable property of the knife edge pattern is that the intensity penetrates some distance into the geometric
#    shadow. Kirchoff diffraction theory predicts that the intensity will be exactly 1/4 of the intensity at the
#    limiting intensity (when the knife edge is completely out of the way).  So we first find the limiting intensity
#    by averaging the tiny and fast wiggles when the knife edge moves far away. That allows us to find the x-position
#    of the 1/4 intensity point and shift the data (taken in x lab coordinates) to line up at u = 0, where u is the
#    dimensionless aperture coordinate. Finally, we scale the theory to match the experimental y intensity at the
#    limiting intensity (not the peak).
    last_100 = current_arr[-100:] # look at the last data points (assumed to be far from knife edge)
    max_last_100 = last_100.max() # They'll be wiggling, so get rough idea of high and low points
    min_last_100 = last_100.min()
    intensity_limit = last_100.mean()  # average last 100 points
    
    y_mod = y_mod * intensity_limit / 2 # scale the model to the data (Note that between limit points on Cornu Spiral
                                        #  the distance is sqrt(2).  Square this to convert E-field amplitude
                                        #  to intensity, hence the factor of 2)   
    print("Intensity limit: ", intensity_limit)
    one_quarter_target = intensity_limit / 4  # Scalar diffraction theory finds the  intensity is 1/4 of the
                                              #   limiting intensity at u = 0.0
    zero_indx = (current_arr >= one_quarter_target) # Find the index of the 1/4 intensity point by marking all values
                                                    #   that are greater than 1/4*I

    quarter_indx = zero_indx.nonzero()[0][0] # get the first such value

    print("quarter index: ", quarter_indx) # find the first element >= 1/4 intensity
    print("quarter index values: ", position_arr[quarter_indx], current_arr[quarter_indx])
    position_arr = position_arr - position_arr[quarter_indx] # shift the pattern to the 1/4 intensity point
    u_exp = position_arr / (1000) * scale_factor
 
fig, ax = plt.subplots()
if (diffraction_type == "Knife_Edge"):
    ax.set_title("Knife Edge, r_s = {:4.3f}, r_o = {:4.3f}".format(r_s, r_o))
if (diffraction_type == "Single_Slit"):
    ax.set_title("Single Strip, r_s = {:4.3f}, r_o = {:4.3f};  a = {:8.6f}".format(r_s, r_o, a_lab))
if (diffraction_type == "Double_Slit"):
    ax.set_title("Double Slit, r_s = {:4.3f}, r_o = {:4.3f};  a = {:8.6f}, d = {:8.6f}".format(r_s, r_o, a_lab, d_lab))

ax.plot(u_exp, current_arr, '.b')  # plot the data as points
#  Now overlay the model, plotted as a solid line
ax.plot(u_mod,y_mod,'-r')

ax.legend(["data", "model"])
ax.set_xlabel("u")  # and show that this is in dimensionless u units
ax.set_ylabel("Intensity")



#  Now consider a finite slit width on the detector.  This will affect all apertures equally
#  du = scalefactor*dx where dx = 25 microns

# pmt_slit_width = 25.00 # μm (stamped on the front).  3 mm tall stainless steel aperture from Lenox Laser
# This is the slit width at the observation plane.  We need to map it onto an equivalent at the location of the aperture.
#  Refer to Fig 11.3.1 in Elmore and Heald, p.407.
#  This is basically a pair of similar triangles:
pmt_slit_width_at_aperture = pmt_slit_width * (r_s / (r_s + r_o))  # needed for model convolution


#print("conversion: ",int(PMT_slit_width*(1e-6)*scale_factor*(10000/20)))
#  (This 10000 over 20 is set by the 10000 points of the model covering +/- 10 u units)
pmt_slit_u_units = int(pmt_slit_width_at_aperture*(1e-6)*scale_factor*(10000/20))+1

print("PMT slit width, in u units: ", pmt_slit_u_units)
# The detector slit is not infinitely narrow, so it will spatially average the data, and smear out the results
#  Here we make a modified model y_mod_convolve to account for this spatial averaging by convolving the aperture
#    with the data
#  First define the shape of the convolution window:

win = signal.windows.tukey(pmt_slit_u_units,alpha=0.7)
#  Now perform the convolution:
y_mod_convolve = signal.convolve(y_mod, win, mode='same')/sum(win)

y_mod_convolve = y_mod_convolve[pmt_slit_u_units//2:-pmt_slit_u_units//2] # truncate by half the convolving window on either side
u_mod = u_mod[pmt_slit_u_units//2:-pmt_slit_u_units//2]
#  Note that we had to clip the ends of the model that stray a half width of the pmt slit out of the u_mod range.  We do this
#    by doing floor division by half the window width
fig, ax = plt.subplots()  # new figure

if (diffraction_type == "Knife_Edge"):
    ax.set_title("Knife Edge (Convolved): r_s = {:4.3f}, r_o = {:4.3f}".format(r_s, r_o))
if (diffraction_type == "Single_Slit"):
    ax.set_title("Single Strip (Convolved): r_s = {:4.3f}, r_o = {:4.3f};  a = {:4.3f} mm".format(r_s, r_o,a_lab))
if (diffraction_type == "Double_Slit"):
    ax.set_title("Double Slit (Convolved): r_s = {:4.3f}, r_o = {:4.3f};  a = {:4.3f} mm, d = {:4.3f} mm".format(r_s, r_o, a_lab, d_lab))


ax.set_xlabel("u")
ax.set_ylabel("Intensity")

ax.plot(u_exp, current_arr, '.b')
ax.plot(u_mod,y_mod_convolve,'-r')
ax.legend(["data", "model"])
plt.show()

Opening file  Fresnel_Diffraction_Single_Slit_Complement_15_Apr_26_01-47-44 (1).csv
Extracting the dictionary of experimental parameters: 

These are the current dictionary values: 
title               	Fresnel Diffraction
r_s                 	0.17
r_o                 	0.473
lamb                	632.8
a_lab               	0.12
d_lab               	[]
start_pos_x         	-1000
end_pos_x           	1000
diffraction_type    	Single_Slit
pmt_slit_width      	25
filename            	Fresnel_Diffraction_Single_Slit_15_Apr_26_01-47-44.csv


Single Slit
position (mm), current (A)

Physical slit width: 0.400 mm
Dimensionless slit width 2.0109
center of mass (centroid) of data:  -0.0050963991424163345
PMT slit width, in u units:  17


In [58]:
# C1: Compare theory and data for all aperture types by reading in the most recent file (the one in the dictionary).  
filename = 'Fresnel_Diffraction_Single_Slit_15_Apr_26_03-32-53.csv'  # This code cell defaults to the present filename.  Edit as needed
                                    #   to pull up and reanalyze a previous data set
#   If you want to call up an old file, simply uncomment the first line and put your file name into the dictionary 
# expt_params['filename'] = 'Fresnel_Diffraction_Knife_Edge_18_Jan_24_01-09-07.csv'  
# (For simplicity, I run this notebook from my desktop, and so the data files get stored in the same directory, and
#   I don't need to specify the full path name.  If you run this notebook from your default user directory, you'll need to 
#   specify the full path name, something like:

#    filename = "C:/Users/tjossem.GRIN.001/Desktop/Fresnel_Diffraction_Knife_Edge_18_Jan_24_01-09-07.csv"

#   Of course, that file no longer exists because ITS has been wiping out all user documents and desktop files after 10 days of
# not logging in....

# One more important aspect of file names in Windows.  Windows thinks the path to the file is 
#   C:\Users\tjossem.GRIN.001\Desktop\Fresnel_Diffraction_Knife_Edge_18_Jan_24_01-09-07.csv
# but for our purposes we'll replace all the forward slashes with backslashes when handling the string in Python.  That's
# because \t \n \r are all interpreted as potential Escape sequences (formatting instructions in C/C++).  So to indicate that 
# they are not Escape sequences, just reverse the \ to /. (Alternatively, replace all single slashes with double slashes,
# indicating that they are 'literals' and not parts of an Escape sequence.)
    
print("Opening file ", filename)
print("Extracting the dictionary of experimental parameters: \n")

with open(filename,'r')as f_in:
    dict_string  = f_in.readline()
    header = f_in.readline()  # get the header as well
expt_params = ast.literal_eval(dict_string)
print_dictionary(expt_params) # print the dictionary to determine what type of model is needed
#  Get diffraction type from dictionary:
diffraction_type = expt_params['diffraction_type']

diffraction_string, plot_style = get_plot_style(diffraction_type)  # update plot header
# Get necessary parameters from the dictionary to calculate model:
r_o = expt_params['r_o']
r_s = expt_params['r_s']
lamb = expt_params['lamb']
a_lab = 0.4
d_lab = expt_params['d_lab']
pmt_slit_width = expt_params['pmt_slit_width'] # get the width of the slit at the PMT for convolution

#  Compute the scale factor that maps lab distances onto dimensionless aperture coordinates:

scale_factor = np.sqrt((2 / (lamb*1.0e-9)) * (1 / r_s + 1 / r_o)) # Elmore and Heald Eq 11.2.12

# Now read in the header and the data:
print(header)
# Read in data using loadtxt, skipping the first two rows that contain the dictionary and the header
position_data, current_data = np.loadtxt(filename,
    delimiter = ",", skiprows = 2, unpack = True) 
position_arr = np.array(position_data)  # convert the text data into (x,y) floating point arrays
current_arr = np.array(current_data)

# In comparing the data to a model, we choose to convert the data to the normalized u coordinates used in the model:
u_exp = (position_arr/(1000) * scale_factor)+0.1685   # convert to u units 
#  The model itself will be computed on a fine grid so that it will plot as a smooth, not chunky, line:
u_mod = np.linspace(-5,5,20000)  # fine grid, 10000 points over 20 u units


#  Compute the model from Kirchoff scalar diffraction theory, depending on the aperture type:
if (diffraction_type == "Knife_Edge"):
    y_mod = np.square(ss.fresnel(u_mod)[1] - (-0.5)) + np.square(ss.fresnel(u_mod)[0] - (-0.5))


elif (diffraction_type == "Single_Slit"):
    a_lab_in_meters = a_lab/1000.0
    a_scaled = scale_factor * a_lab_in_meters  # scale the lab distances in order to put into dimensionless u coordinates
    #  Now the diffraction integral integrates from the left edge to the right edge of the single slit
    y_mod = np.square(ss.fresnel(u_mod + a_scaled/2)[1] - ss.fresnel(u_mod - a_scaled/2)[1]) +\
        np.square(ss.fresnel(u_mod + a_scaled/2)[0] - ss.fresnel(u_mod - a_scaled/2)[0])
    print("Physical slit width: {:5.3f} mm".format(a_lab))
    print("Dimensionless slit width {:5.4f}".format(a_scaled))


elif (diffraction_type == "Double_Slit"):
    a_lab_in_meters = expt_params['a_lab']/1000.0
    d_lab_in_meters = expt_params['d_lab']/1000.0
    a_scaled = scale_factor * a_lab_in_meters
    d_scaled = scale_factor * d_lab_in_meters
    #  Now the diffraction integral integrates from the left edge to the right edge of each slit
    y_mod = np.square(ss.fresnel(u_mod + d_scaled/2 + a_scaled/2)[1] - ss.fresnel(u_mod + d_scaled/2 - a_scaled/2)[1] + \
                      ss.fresnel(u_mod - d_scaled/2 + a_scaled/2)[1]\
                   - ss.fresnel(u_mod - d_scaled/2 - a_scaled/2)[1])\
    + np.square(ss.fresnel(u_mod + d_scaled/2 + a_scaled/2)[0] - ss.fresnel(u_mod + d_scaled/2 - a_scaled/2)[0] 
             + ss.fresnel(u_mod - d_scaled/2 + a_scaled/2)[0] - ss.fresnel(u_mod - d_scaled/2 - a_scaled/2)[0])
    print("Physical slit width: {:5.3f} mm and slit spacing {:5.3f} mm".format(a_lab, d_lab))
    print("Dimensionless slit width {:5.4f} and slit spacing {:5.4f}".format(a_scaled, d_scaled))

# Now clean up the data a little if necessary.  This step is optional, and is probably not needed.  It averages 
#   the first and last 10 data points in order to get an estimate of the baseline
#baseline = np.sum(current_arr[0:10]+current_arr[-10:])/20
# For now, assume the baseline is zero:
baseline = 0.00  # For really wide patterns you may not get a good baseline here.  If you keep the room lights off
                 #   you should find that the baseline is essentially zero and so for now we fix it that way
current_arr=current_arr-baseline  # optional vertical adjustment

# Now find the center of the pattern (centroid) by doing a 'center of mass' calculation, which multiplies
#   the signal times its position, then divides by the sum of all the signals.

center_of_mass = (current_arr*position_arr).sum()/current_arr.sum()  # centroid
print("center of mass (centroid) of data: ", center_of_mass)
position_arr = position_arr-center_of_mass    # subtract the centroid so as to center the data on the theory

#  Now that we have the model, do some scaling vertically so as to normalize the arbitary intensity.  For the single
#    and double slits, we simply match the vertical heights:

if (diffraction_type != "Knife_Edge"):    
    max_val_mod = y_mod.max()  # find max value of model to use for vertical scaling
    max_val = current_arr.max()  # find max value of data to use for vertical scaling
    y_mod = y_mod*max_val/max_val_mod  # scale the model to match the height of the data

else:  #  Here for the knife edge
#  A remarkable property of the knife edge pattern is that the intensity penetrates some distance into the geometric
#    shadow. Kirchoff diffraction theory predicts that the intensity will be exactly 1/4 of the intensity at the
#    limiting intensity (when the knife edge is completely out of the way).  So we first find the limiting intensity
#    by averaging the tiny and fast wiggles when the knife edge moves far away. That allows us to find the x-position
#    of the 1/4 intensity point and shift the data (taken in x lab coordinates) to line up at u = 0, where u is the
#    dimensionless aperture coordinate. Finally, we scale the theory to match the experimental y intensity at the
#    limiting intensity (not the peak).
    last_100 = current_arr[-100:] # look at the last data points (assumed to be far from knife edge)
    max_last_100 = last_100.max() # They'll be wiggling, so get rough idea of high and low points
    min_last_100 = last_100.min()
    intensity_limit = last_100.mean()  # average last 100 points
    
    y_mod = y_mod * intensity_limit / 2 # scale the model to the data (Note that between limit points on Cornu Spiral
                                        #  the distance is sqrt(2).  Square this to convert E-field amplitude
                                        #  to intensity, hence the factor of 2)   
    print("Intensity limit: ", intensity_limit)
    one_quarter_target = intensity_limit / 4  # Scalar diffraction theory finds the  intensity is 1/4 of the
                                              #   limiting intensity at u = 0.0
    zero_indx = (current_arr >= one_quarter_target) # Find the index of the 1/4 intensity point by marking all values
                                                    #   that are greater than 1/4*I

    quarter_indx = zero_indx.nonzero()[0][0] # get the first such value

    print("quarter index: ", quarter_indx) # find the first element >= 1/4 intensity
    print("quarter index values: ", position_arr[quarter_indx], current_arr[quarter_indx])
    position_arr = position_arr - position_arr[quarter_indx] # shift the pattern to the 1/4 intensity point
    u_exp = position_arr / (1000) * scale_factor
 
fig, ax = plt.subplots()
if (diffraction_type == "Knife_Edge"):
    ax.set_title("Knife Edge, r_s = {:4.3f}, r_o = {:4.3f}".format(r_s, r_o))
if (diffraction_type == "Single_Slit"):
    ax.set_title("Single Slit, r_s = {:4.3f}, r_o = {:4.3f};  a = {:8.6f}".format(r_s, r_o, a_lab))
if (diffraction_type == "Double_Slit"):
    ax.set_title("Double Slit, r_s = {:4.3f}, r_o = {:4.3f};  a = {:8.6f}, d = {:8.6f}".format(r_s, r_o, a_lab, d_lab))

ax.plot(u_exp, current_arr, '.b')  # plot the data as points
#  Now overlay the model, plotted as a solid line
ax.plot(u_mod,y_mod,'-r')

ax.legend(["data", "model"])
ax.set_xlabel("u")  # and show that this is in dimensionless u units
ax.set_ylabel("Intensity")



#  Now consider a finite slit width on the detector.  This will affect all apertures equally
#  du = scalefactor*dx where dx = 25 microns

# pmt_slit_width = 25.00 # μm (stamped on the front).  3 mm tall stainless steel aperture from Lenox Laser
# This is the slit width at the observation plane.  We need to map it onto an equivalent at the location of the aperture.
#  Refer to Fig 11.3.1 in Elmore and Heald, p.407.
#  This is basically a pair of similar triangles:
pmt_slit_width_at_aperture = pmt_slit_width * (r_s / (r_s + r_o))  # needed for model convolution


#print("conversion: ",int(PMT_slit_width*(1e-6)*scale_factor*(10000/20)))
#  (This 10000 over 20 is set by the 10000 points of the model covering +/- 10 u units)
pmt_slit_u_units = int(pmt_slit_width_at_aperture*(1e-6)*scale_factor*(10000/20))+1

print("PMT slit width, in u units: ", pmt_slit_u_units)
# The detector slit is not infinitely narrow, so it will spatially average the data, and smear out the results
#  Here we make a modified model y_mod_convolve to account for this spatial averaging by convolving the aperture
#    with the data
#  First define the shape of the convolution window:

win = signal.windows.tukey(pmt_slit_u_units,alpha=0.7)
#  Now perform the convolution:
y_mod_convolve = signal.convolve(y_mod, win, mode='same')/sum(win)

y_mod_convolve = y_mod_convolve[pmt_slit_u_units//2:-pmt_slit_u_units//2] # truncate by half the convolving window on either side
u_mod = u_mod[pmt_slit_u_units//2:-pmt_slit_u_units//2]
#  Note that we had to clip the ends of the model that stray a half width of the pmt slit out of the u_mod range.  We do this
#    by doing floor division by half the window width
fig, ax = plt.subplots()  # new figure

if (diffraction_type == "Knife_Edge"):
    ax.set_title("Knife Edge (Convolved): r_s = {:4.3f}, r_o = {:4.3f}".format(r_s, r_o))
if (diffraction_type == "Single_Slit"):
    ax.set_title("Single Slit (Convolved): r_s = {:4.3f}, r_o = {:4.3f};  a = {:4.3f} mm".format(r_s, r_o,a_lab))
if (diffraction_type == "Double_Slit"):
    ax.set_title("Double Slit (Convolved): r_s = {:4.3f}, r_o = {:4.3f};  a = {:4.3f} mm, d = {:4.3f} mm".format(r_s, r_o, a_lab, d_lab))


ax.set_xlabel("u")
ax.set_ylabel("Intensity")

ax.plot(u_exp, current_arr, '.b')
ax.plot(u_mod,y_mod_convolve,'-r')
ax.legend(["data", "model"])
plt.show()

Opening file  Fresnel_Diffraction_Single_Slit_15_Apr_26_03-32-53.csv
Extracting the dictionary of experimental parameters: 

These are the current dictionary values: 
title               	Fresnel Diffraction
r_s                 	0.17
r_o                 	0.473
lamb                	632.8
a_lab               	0.4
d_lab               	[]
start_pos_x         	-1000
end_pos_x           	1000
diffraction_type    	Single_Slit
pmt_slit_width      	25
filename            	Fresnel_Diffraction_Single_Slit_15_Apr_26_03-32-53.csv


Single Slit
position (mm), current (A)

Physical slit width: 0.400 mm
Dimensionless slit width 2.0109
center of mass (centroid) of data:  -0.03605502887884068
PMT slit width, in u units:  17


In [60]:
# C1: Compare theory and data for all aperture types by reading in the most recent file (the one in the dictionary).  
filename1 = 'Fresnel_Diffraction_Single_Slit_15_Apr_26_03-32-53.csv'  # This code cell defaults to the present filename.  Edit as needed
                                    #   to pull up and reanalyze a previous data set
filename2 = 'Fresnel_Diffraction_Single_Slit_Complement_15_Apr_26_01-47-44 (1).csv'
#   If you want to call up an old file, simply uncomment the first line and put your file name into the dictionary 
# expt_params['filename'] = 'Fresnel_Diffraction_Knife_Edge_18_Jan_24_01-09-07.csv'  
# (For simplicity, I run this notebook from my desktop, and so the data files get stored in the same directory, and
#   I don't need to specify the full path name.  If you run this notebook from your default user directory, you'll need to 
#   specify the full path name, something like:

#    filename = "C:/Users/tjossem.GRIN.001/Desktop/Fresnel_Diffraction_Knife_Edge_18_Jan_24_01-09-07.csv"

#   Of course, that file no longer exists because ITS has been wiping out all user documents and desktop files after 10 days of
# not logging in....

# One more important aspect of file names in Windows.  Windows thinks the path to the file is 
#   C:\Users\tjossem.GRIN.001\Desktop\Fresnel_Diffraction_Knife_Edge_18_Jan_24_01-09-07.csv
# but for our purposes we'll replace all the forward slashes with backslashes when handling the string in Python.  That's
# because \t \n \r are all interpreted as potential Escape sequences (formatting instructions in C/C++).  So to indicate that 
# they are not Escape sequences, just reverse the \ to /. (Alternatively, replace all single slashes with double slashes,
# indicating that they are 'literals' and not parts of an Escape sequence.)
    
print("Opening file ", filename1)
print("Opening file ", filename2)
print("Extracting the dictionary of experimental parameters: \n")

with open(filename,'r')as f_in:
    dict_string  = f_in.readline()
    header = f_in.readline()  # get the header as well
expt_params = ast.literal_eval(dict_string)
print_dictionary(expt_params) # print the dictionary to determine what type of model is needed
#  Get diffraction type from dictionary:
diffraction_type = 'Single_Slit'

diffraction_string, plot_style = get_plot_style(diffraction_type)  # update plot header
# Get necessary parameters from the dictionary to calculate model:
r_o = 0.473
r_s = 0.170
lamb = 632.8
a_lab = 0.4
d_lab = []
pmt_slit_width = 25 # get the width of the slit at the PMT for convolution

#  Compute the scale factor that maps lab distances onto dimensionless aperture coordinates:

scale_factor = np.sqrt((2 / (lamb*1.0e-9)) * (1 / r_s + 1 / r_o)) # Elmore and Heald Eq 11.2.12

# Now read in the header and the data:
print(header)
# Read in data using loadtxt, skipping the first two rows that contain the dictionary and the header
position_dataSl, current_dataSl = np.loadtxt(filename1,
    delimiter = ",", skiprows = 2, unpack = True) 
position_arrSl = np.array(position_dataSl)  # convert the text data into (x,y) floating point arrays
current_arrSl = np.array(current_dataSl)

# Read in data using loadtxt, skipping the first two rows that contain the dictionary and the header
position_dataSt, current_dataSt = np.loadtxt(filename2,
    delimiter = ",", skiprows = 2, unpack = True) 
position_arrSt = np.array(position_dataSt)  # convert the text data into (x,y) floating point arrays
current_arrSt = np.array(current_dataSt)


# In comparing the data to a model, we choose to convert the data to the normalized u coordinates used in the model:
u_expSl = (position_arrSl/(1000) * scale_factor)+0.1685   # convert to u units
u_expSt = (position_arrSt/(1000) * scale_factor)   # convert to u units 
#  The model itself will be computed on a fine grid so that it will plot as a smooth, not chunky, line:
u_mod = np.linspace(-5,5,20000)  # fine grid, 10000 points over 20 u units


#  Compute the model from Kirchoff scalar diffraction theory, depending on the aperture type:
if (diffraction_type == "Knife_Edge"):
    y_mod = np.square(ss.fresnel(u_mod)[1] - (-0.5)) + np.square(ss.fresnel(u_mod)[0] - (-0.5))


elif (diffraction_type == "Single_Slit"):
    a_lab_in_meters = a_lab/1000.0
    a_scaled = scale_factor * a_lab_in_meters  # scale the lab distances in order to put into dimensionless u coordinates
    #  Now the diffraction integral integrates from the left edge to the right edge of the single slit
    y_mod = np.square(1) +\
        np.square(1)
    print("Physical slit width: {:5.3f} mm".format(a_lab))
    print("Dimensionless slit width {:5.4f}".format(a_scaled))


elif (diffraction_type == "Double_Slit"):
    a_lab_in_meters = expt_params['a_lab']/1000.0
    d_lab_in_meters = expt_params['d_lab']/1000.0
    a_scaled = scale_factor * a_lab_in_meters
    d_scaled = scale_factor * d_lab_in_meters
    #  Now the diffraction integral integrates from the left edge to the right edge of each slit
    y_mod = np.square(ss.fresnel(u_mod + d_scaled/2 + a_scaled/2)[1] - ss.fresnel(u_mod + d_scaled/2 - a_scaled/2)[1] + \
                      ss.fresnel(u_mod - d_scaled/2 + a_scaled/2)[1]\
                   - ss.fresnel(u_mod - d_scaled/2 - a_scaled/2)[1])\
    + np.square(ss.fresnel(u_mod + d_scaled/2 + a_scaled/2)[0] - ss.fresnel(u_mod + d_scaled/2 - a_scaled/2)[0] 
             + ss.fresnel(u_mod - d_scaled/2 + a_scaled/2)[0] - ss.fresnel(u_mod - d_scaled/2 - a_scaled/2)[0])
    print("Physical slit width: {:5.3f} mm and slit spacing {:5.3f} mm".format(a_lab, d_lab))
    print("Dimensionless slit width {:5.4f} and slit spacing {:5.4f}".format(a_scaled, d_scaled))

# Now clean up the data a little if necessary.  This step is optional, and is probably not needed.  It averages 
#   the first and last 10 data points in order to get an estimate of the baseline
#baseline = np.sum(current_arr[0:10]+current_arr[-10:])/20
# For now, assume the baseline is zero:
baseline = 0.00  # For really wide patterns you may not get a good baseline here.  If you keep the room lights off
                 #   you should find that the baseline is essentially zero and so for now we fix it that way
current_arrSl=current_arrSl-baseline  # optional vertical adjustment

# Now find the center of the pattern (centroid) by doing a 'center of mass' calculation, which multiplies
#   the signal times its position, then divides by the sum of all the signals.

center_of_massSl = (current_arrSl*position_arrSl).sum()/current_arrSl.sum()  # centroid
print("center of mass (centroid) of data: ", center_of_massSl)
position_arrSl = position_arrSl-center_of_massSl    # subtract the centroid so as to center the data on the theory

current_arrSt=current_arrSt-baseline  # optional vertical adjustment

# Now find the center of the pattern (centroid) by doing a 'center of mass' calculation, which multiplies
#   the signal times its position, then divides by the sum of all the signals.

center_of_massSt = (current_arrSt*position_arrSt).sum()/current_arrSt.sum()  # centroid
print("center of mass (centroid) of data: ", center_of_massSt)
position_arrSt = position_arrSt-center_of_massSt    # subtract the centroid so as to center the data on the theory


#  Now that we have the model, do some scaling vertically so as to normalize the arbitary intensity.  For the single
#    and double slits, we simply match the vertical heights:

if (diffraction_type != "Knife_Edge"):    
    max_val_mod = y_mod.max()  # find max value of model to use for vertical scaling
    max_val = current_arr.max()  # find max value of data to use for vertical scaling
    y_mod = y_mod*max_val/max_val_mod  # scale the model to match the height of the data

else:  #  Here for the knife edge
#  A remarkable property of the knife edge pattern is that the intensity penetrates some distance into the geometric
#    shadow. Kirchoff diffraction theory predicts that the intensity will be exactly 1/4 of the intensity at the
#    limiting intensity (when the knife edge is completely out of the way).  So we first find the limiting intensity
#    by averaging the tiny and fast wiggles when the knife edge moves far away. That allows us to find the x-position
#    of the 1/4 intensity point and shift the data (taken in x lab coordinates) to line up at u = 0, where u is the
#    dimensionless aperture coordinate. Finally, we scale the theory to match the experimental y intensity at the
#    limiting intensity (not the peak).
    last_100 = current_arr[-100:] # look at the last data points (assumed to be far from knife edge)
    max_last_100 = last_100.max() # They'll be wiggling, so get rough idea of high and low points
    min_last_100 = last_100.min()
    intensity_limit = last_100.mean()  # average last 100 points
    
    y_mod = y_mod * intensity_limit / 2 # scale the model to the data (Note that between limit points on Cornu Spiral
                                        #  the distance is sqrt(2).  Square this to convert E-field amplitude
                                        #  to intensity, hence the factor of 2)   
    print("Intensity limit: ", intensity_limit)
    one_quarter_target = intensity_limit / 4  # Scalar diffraction theory finds the  intensity is 1/4 of the
                                              #   limiting intensity at u = 0.0
    zero_indx = (current_arr >= one_quarter_target) # Find the index of the 1/4 intensity point by marking all values
                                                    #   that are greater than 1/4*I

    quarter_indx = zero_indx.nonzero()[0][0] # get the first such value

    print("quarter index: ", quarter_indx) # find the first element >= 1/4 intensity
    print("quarter index values: ", position_arr[quarter_indx], current_arr[quarter_indx])
    position_arr = position_arr - position_arr[quarter_indx] # shift the pattern to the 1/4 intensity point
    u_exp = position_arr / (1000) * scale_factor
 
fig, ax = plt.subplots()
if (diffraction_type == "Knife_Edge"):
    ax.set_title("Knife Edge, r_s = {:4.3f}, r_o = {:4.3f}".format(r_s, r_o))
if (diffraction_type == "Single_Slit"):
    ax.set_title("Single Slit, r_s = {:4.3f}, r_o = {:4.3f};  a = {:8.6f}".format(r_s, r_o, a_lab))
if (diffraction_type == "Double_Slit"):
    ax.set_title("Double Slit, r_s = {:4.3f}, r_o = {:4.3f};  a = {:8.6f}, d = {:8.6f}".format(r_s, r_o, a_lab, d_lab))

ax.plot(u_exp, current_arr, '.b')  # plot the data as points
#  Now overlay the model, plotted as a solid line
ax.plot(u_mod,y_mod,'-r')

ax.legend(["data", "model"])
ax.set_xlabel("u")  # and show that this is in dimensionless u units
ax.set_ylabel("Intensity")



#  Now consider a finite slit width on the detector.  This will affect all apertures equally
#  du = scalefactor*dx where dx = 25 microns

# pmt_slit_width = 25.00 # μm (stamped on the front).  3 mm tall stainless steel aperture from Lenox Laser
# This is the slit width at the observation plane.  We need to map it onto an equivalent at the location of the aperture.
#  Refer to Fig 11.3.1 in Elmore and Heald, p.407.
#  This is basically a pair of similar triangles:
pmt_slit_width_at_aperture = pmt_slit_width * (r_s / (r_s + r_o))  # needed for model convolution


#print("conversion: ",int(PMT_slit_width*(1e-6)*scale_factor*(10000/20)))
#  (This 10000 over 20 is set by the 10000 points of the model covering +/- 10 u units)
pmt_slit_u_units = int(pmt_slit_width_at_aperture*(1e-6)*scale_factor*(10000/20))+1

print("PMT slit width, in u units: ", pmt_slit_u_units)
# The detector slit is not infinitely narrow, so it will spatially average the data, and smear out the results
#  Here we make a modified model y_mod_convolve to account for this spatial averaging by convolving the aperture
#    with the data
#  First define the shape of the convolution window:

win = signal.windows.tukey(pmt_slit_u_units,alpha=0.7)
#  Now perform the convolution:
y_mod_convolve = signal.convolve(y_mod, win, mode='same')/sum(win)

y_mod_convolve = y_mod_convolve[pmt_slit_u_units//2:-pmt_slit_u_units//2] # truncate by half the convolving window on either side
u_mod = u_mod[pmt_slit_u_units//2:-pmt_slit_u_units//2]
#  Note that we had to clip the ends of the model that stray a half width of the pmt slit out of the u_mod range.  We do this
#    by doing floor division by half the window width
fig, ax = plt.subplots()  # new figure


if (diffraction_type == "Knife_Edge"):
    ax.set_title("Knife Edge (Convolved): r_s = {:4.3f}, r_o = {:4.3f}".format(r_s, r_o))
if (diffraction_type == "Single_Slit"):
    ax.set_title("Single Slit (Convolved): r_s = {:4.3f}, r_o = {:4.3f};  a = {:4.3f} mm".format(r_s, r_o,a_lab))
if (diffraction_type == "Double_Slit"):
    ax.set_title("Double Slit (Convolved): r_s = {:4.3f}, r_o = {:4.3f};  a = {:4.3f} mm, d = {:4.3f} mm".format(r_s, r_o, a_lab, d_lab))


ax.set_xlabel("u")
ax.set_ylabel("Intensity")

ax.plot(u_expSl, current_arrSl, '.b')
ax.plot(u_mod,y_mod_convolve,'-r')
ax.legend(["data", "model"])
plt.show()

Opening file  Fresnel_Diffraction_Single_Slit_15_Apr_26_03-32-53.csv
Opening file  Fresnel_Diffraction_Single_Slit_Complement_15_Apr_26_01-47-44 (1).csv
Extracting the dictionary of experimental parameters: 

These are the current dictionary values: 
title               	Fresnel Diffraction
r_s                 	0.17
r_o                 	0.473
lamb                	632.8
a_lab               	0.4
d_lab               	[]
start_pos_x         	-1000
end_pos_x           	1000
diffraction_type    	Single_Slit
pmt_slit_width      	25
filename            	Fresnel_Diffraction_Single_Slit_15_Apr_26_03-32-53.csv


Single Slit
position (mm), current (A)

Physical slit width: 0.400 mm
Dimensionless slit width 2.0109
center of mass (centroid) of data:  -0.03605502887884068
center of mass (centroid) of data:  -0.0050963991424163345


ValueError: x and y must have same first dimension, but have shapes (20000,) and (1,)

###  Save a figure 

In [ ]:
### Want to save a plot of your data along with the model plot?  Get it in focus and then say:

plt.savefig("Your descriptive figure name here.png")  # (or .tiff, or .jpg) Then copy and paste into a Word document

### C2: Alternative (optional) model to calculate Fresnel integrals for the knife edge, modified by Gaussian beam profile. This takes a little longer because it directly integrates the Fresnel integrands multiplied by a Gaussian beam shape (a nice approximation to the Airy disk).  

In [ ]:
# C2:  Alternative model to include effect of Gaussian beam
if (expt_params['diffraction_type'] != 'Knife_Edge'):
    print("This model is only for the knife edge")
position_arr = np.array(position_data) # turn lists into arrays for easier processing
current_arr = np.array(current_data)
y_offset = current_arr[0] # use first data point to capture any DC offset on signal

current_arr = current_arr - y_offset  # subtract offset from data
last_100 = current_arr[-100:] # look at the last data points (assumed to be far from knife edge)

intensity_limit = last_100.mean() # average last points
print("Intensity limit: ", intensity_limit)
one_quarter_target = intensity_limit / 4
zero_indx = (current_arr >= one_quarter_target) # Find the index of the 1/4 intensity point by marking all values
                                                #   that are greater than 1/4I

quarter_indx = zero_indx.nonzero()[0][0]

print("quarter index: ", quarter_indx) # find the first element >= 1/4 intensity
print("quarter index values: ", position_arr[quarter_indx], current_arr[quarter_indx])
position_arr = position_arr - position_arr[quarter_indx] # shift the pattern to the 1/4 intensity point

u_exp = position_arr / (1000) * scale_factor

u_mod = np.linspace(-10, 10, 10000) # fine grid for plotting model

fig, ax = plt.subplots()
w = 18  #  an unscaled Gaussian width parameter that models the falloff in intensity as you get farther from centerline
        #  smaller values indicate faster falloff.  This number is highly dependent on whether you have optimized the 
        #    z-position of the spatial filter in front of the laser
ax.set_title("Knife Edge, including effect of Gaussian beam profile, w = {:}".format(w))
fcb = []
fsb = []
# A very inefficient way to (re)calculate the Fresnel integrals, taking the intensity envelope into account.
#   This assumes our slit isn't very tall so a slice through the Gaussian is approximated by its radial distance 
#     from the center.  Inefficient because we could use a cumulative sum of the integrands.
for umax in u_mod:
    fcb = np.append(fcb, integrate.quad(lambda t: np.cos(0.5 * np.pi * t**2) * np.exp(-(t / w)**2), 0, umax)[0])
    fsb = np.append(fsb, integrate.quad(lambda t: np.sin(0.5 * np.pi * t**2) * np.exp(-(t / w)**2), 0, umax)[0])
y_mod = np.square(fsb - (-0.5)) + np.square(fcb - (-0.5))

gauss = np.exp(-(u_mod / w)**2)     # capture the shape of the Gaussian
y_mod = y_mod * intensity_limit / 2 # scale the model to the data

ax.plot(u_mod, y_mod, '-r')                     # plot model as connected lines
ax.plot(u_exp, current_arr, '.b', ms = 3)       # plot data as small points
ax.plot(u_mod, intensity_limit * gauss, '--g')  # plot the gaussian shape
ax.legend(["model", "data", "profile"])
ax.plot(u_exp[-100:], current_arr[-100:], '-g') # just to see if we are capturing the final wiggles
ax.plot(position_arr[quarter_indx], current_arr[quarter_indx], '*k', ms = 10)
ax.set_xlabel("u")
ax.set_ylabel("Intensity")

plt.show()

pmt_slit_u_units = int(pmt_slit_width_at_aperture * (1e-6) * scale_factor * (10000 / 20)) + 1
print("PMT slit width, in u units: ", pmt_slit_u_units)
win = signal.windows.tukey(pmt_slit_u_units, alpha = 0.7)

y_mod = signal.convolve(y_mod, win, mode = 'same') / win.sum()

fig, ax = plt.subplots()
ax.set_title("Knife Edge (Convolved with a Gaussian beam profile)")
ax.set_xlabel("u")
ax.set_ylabel("Intensity")
ax.plot(u_mod[:-100], y_mod[:-100],'-r')
ax.plot(u_exp, current_arr, '.b')
ax.legend(["model", "data"])
plt.show()

In [ ]:
#  Extra refinement:  include the effect of the Gaussian beam profile and the width of the detector slit

pmt_slit_width_at_aperture = pmt_slit_width * (r_s / (r_s + r_o))  # needed for model convolution
u_mod = np.linspace(-10, 10, 10000) # fine grid for plotting model (need to reset because a window width is missing)

#print("conversion: ",int(PMT_slit_width*(1e-6)*scale_factor*(10000/20)))
#  (This 10000 over 20 is set by the 10000 points of the model covering +/- 10 u units)
pmt_slit_u_units = int(pmt_slit_width_at_aperture*(1e-6)*scale_factor*(10000/20))+1

print("PMT slit width, in u units: ", pmt_slit_u_units)
# The detector slit is not infinitely narrow, so it will spatially average the data, and smear out the results
#  Here we make a modified model y_mod_convolve to account for this spatial averaging by convolving the aperture
#    with the data
#  First define the shape of the convolution window:

win = signal.windows.tukey(pmt_slit_u_units,alpha=0.7)
#  Now perform the convolution:
y_mod_convolve = signal.convolve(y_mod, win, mode='same')/sum(win)

y_mod_convolve = y_mod_convolve[pmt_slit_u_units//2:-pmt_slit_u_units//2] # truncate by half the convolving window on either side
u_mod = u_mod[pmt_slit_u_units//2:-pmt_slit_u_units//2]
#  Note that we had to clip the ends of the model that stray a half width of the pmt slit out of the u_mod range.  We do this
#    by doing floor division by half the window width
fig, ax = plt.subplots()  # new figure
plt.suptitle(filename)
ax.set_title("Knife Edge with Gaussian beam (Convolved): r_s = {:4.3f}, r_o = {:4.3f}".format(r_s, r_o))

ax.set_xlabel("u")
ax.set_ylabel("Intensity")

ax.plot(u_exp, current_arr, '.b')
ax.plot(u_mod,y_mod_convolve,'-r')
ax.legend(["data", "model"])
plt.show()

### Here to save the most recent figure as a .png file, in the current zoomed image

In [ ]:
print(filename)
figname = filename[:-4]
print("Saving as a PNG (the default) under file name: ", figname)
plt.savefig(figname)


### Announcement:  when you are done with the experiment, remember to hover over the Jupyter  Notebook window and hit Ctrl-C twice to quit the kernel and free up the USB resources for the next group.  Thanks!